In [14]:
import pandas as pd
import numpy as np
import os
import time
import json
import uuid
import sys
from codecarbon import OfflineEmissionsTracker
from memory_profiler import memory_usage
from avatars.manager import Manager
from avatars.models import JobKind

# Path setup
# sys.path.append("/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/")
# from SynOmics.utils.monitoring import monitor_resources as _monitor_resources

def run_avatars_block_benchmark(i, seed, k, manager, output_path):
    """
    Thực hiện và đo lường tài nguyên (RAM, Emission, Time) của đúng 1 block.
    """
    
    # 1. Payload thực thi
    def block_payload():
        data = pd.read_csv(f"avatars/original_blocks/original_block_{i}.csv", index_col=False)
        data_clean = data.drop(columns=["Patient"])
        n_feats = data_clean.shape[1]
        
        job_name = f"Block_{i}_k{k}_{seed}_" + str(uuid.uuid4())
        runner = manager.create_runner(job_name, seed=seed)
        table_name = f"block_{i}_k{k}_{seed}_" + str(uuid.uuid4())
        
        runner.add_table(table_name, data_clean)
        runner.set_parameters(table_name, k=k)
        
        runner.run(jobs_to_run=[JobKind.standard])
        
        synthetic_df = runner.sensitive_unshuffled(table_name)
        synthetic_df.to_csv(f"{output_path}/synthetic_block_{i}.csv", index=False)
        
        return n_feats

    # 2. Setup Tracker
    tracker = OfflineEmissionsTracker(
        tracking_mode='process',
        country_iso_code="VNM",
        output_dir=output_path,
        output_file=f"emissions_block_{i}.csv"
    )

    tracker.start()
    start_time = time.perf_counter() # Bắt đầu đo thời gian
    try:
        # 3. Đo Memory history
        mem_samples, n_features = memory_usage(
            (block_payload,), 
            interval=0.2, 
            retval=True
        )
        end_time = time.perf_counter() # Kết thúc đo thời gian
        
        duration = end_time - start_time
        peak_ram = max(mem_samples)
        
        # 4. Lưu lịch sử Memory của block i
        history_df = pd.DataFrame({
            'time_step_sec': [j * 0.2 for j in range(len(mem_samples))],
            'mem_usage_mib': mem_samples
        })
        history_df.to_csv(os.path.join(output_path, f"memory_history_block_{i}.csv"), index=False)
        
        print(f"Block {i} | Time: {duration:.2f}s | Peak RAM: {peak_ram:.2f} MiB")
        
        return {
            'block_id': i,
            'duration_sec': duration,
            'peak_ram_mib': peak_ram,
            'n_features': n_features
        }

    finally:
        tracker.stop()

In [16]:
# --- Main Script ---
# if __name__ == "__main__":
url = os.environ.get("AVATAR_BASE_API_URL", "https://www.octopize.app/api")
username = "the-chuong.trinh@cea.fr"
password = "ttcAVATARS#123"

seed = 42
k = 5
output_dir = f"Benchmark/avatars/synthetic_blocks_k{k}_{seed}"
os.makedirs(output_dir, exist_ok=True)

manager = Manager(base_url=url)
manager.authenticate(username, password, should_verify_compatibility=False)

with open("avatars/cluster_final.json", "r") as f:
    cluster_features = json.load(f)

all_stats = [] # Lưu trữ kết quả tổng hợp

print(f"=== Starting Avatars Anonymization Benchmark (Seed: {seed}) ===")
# blocks = [13]
for i in range(len(cluster_features)):
# for i in blocks:
    print(f"--- Profiling Block {i} / {len(cluster_features)-1} ---")
    try:
        # Chạy benchmark và nhận kết quả thống kê
        stats = run_avatars_block_benchmark(i, seed, k, manager, output_dir)
        all_stats.append(stats)
        
        # Sleep logic (Ngoài vùng đo)
        if stats['n_features'] >= 3000:
            time.sleep(1800)
        else:
            time.sleep(30)
            
    except Exception as e:
        print(f"Error processing block {i}: {e}")
        continue

# 5. Xuất file tổng hợp cuối cùng cho seed này
summary_df = pd.DataFrame(all_stats)
summary_df.to_csv(f"{output_dir}/summary_stats_seed_{seed}.csv", index=False)

print(f"\n=== Benchmark Complete. Summary saved to {output_dir}/summary_stats_seed_{seed}.csv ===")

2026-02-05 00:53:20 [debug    ] datauploader initialized       storage_endpoint_url=https://www.octopize.app/storage
2026-02-05 00:53:20 [debug    ] ApiClient initialized          base_api_url=https://www.octopize.app/api


[codecarbon INFO @ 00:53:20] offline tracker init
[codecarbon WARNING @ 00:53:20] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 00:53:20] [setup] RAM Tracking...
[codecarbon INFO @ 00:53:20] [setup] CPU Tracking...
[codecarbon INFO @ 00:53:20] Tracking Intel CPU via RAPL interface
[codecarbon INFO @ 00:53:20] 	RAPL - Using 6 package domain(s) for CPU power measurement
[codecarbon INFO @ 00:53:20] 	RAPL - Selected 2 unique RAPL domain(s) after deduplication
[codecarbon INFO @ 00:53:20] 	RAPL - Monitoring domain 'package-1' (displayed as 'Processor Energy Delta_0(kWh)') via MSR at /sys/class/powercap/intel-rapl/subsystem/intel-rapl/intel-rapl:1/energy_uj
[codecarbon INFO @ 00:53:20] 	RAPL - Monitoring domain 'package-0' (displayed as 'Processor Energy Delta_1(kWh)') via MSR at /sys/class/powercap/intel-rapl/subsystem/intel-rapl/intel-rapl:0/energy_uj
[codecarbon INFO @ 00:53:20] [setup] GPU Tracking...
[codecarbon INFO @ 00:53:20] Tracking Nvidi

=== Starting Avatars Anonymization Benchmark (Seed: 42) ===
--- Profiling Block 0 / 39 ---


[codecarbon INFO @ 00:53:27] Energy consumed for RAM : 0.000020 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 00:53:27] Delta energy consumed for CPU with intel_rapl : 0.000232 kWh, power : 14.808859094040912 W
[codecarbon INFO @ 00:53:27] Energy consumed for All CPU : 0.000232 kWh
[codecarbon INFO @ 00:53:27] Energy consumed for all GPUs : 0.000080 kWh. Total GPU Power : 41.08484341792541 W
[codecarbon INFO @ 00:53:27] 0.000332 kWh of electricity and 0.000000 L of water were used since the beginning.


Block 0 | Time: 7.00s | Peak RAM: 234.87 MiB


...

Block 38 | Time: 12.61s | Peak RAM: 232.49 MiB


[codecarbon INFO @ 01:14:59] offline tracker init
[codecarbon WARNING @ 01:14:59] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 01:14:59] [setup] RAM Tracking...
[codecarbon INFO @ 01:14:59] [setup] CPU Tracking...
[codecarbon INFO @ 01:14:59] Tracking Intel CPU via RAPL interface
[codecarbon INFO @ 01:14:59] 	RAPL - Using 6 package domain(s) for CPU power measurement
[codecarbon INFO @ 01:14:59] 	RAPL - Selected 2 unique RAPL domain(s) after deduplication
[codecarbon INFO @ 01:14:59] 	RAPL - Monitoring domain 'package-1' (displayed as 'Processor Energy Delta_0(kWh)') via MSR at /sys/class/powercap/intel-rapl/subsystem/intel-rapl/intel-rapl:1/energy_uj
[codecarbon INFO @ 01:14:59] 	RAPL - Monitoring domain 'package-0' (displayed as 'Processor Energy Delta_1(kWh)') via MSR at /sys/class/powercap/intel-rapl/subsystem/intel-rapl/intel-rapl:0/energy_uj
[codecarbon INFO @ 01:14:59] [setup] GPU Tracking...
[codecarbon INFO @ 01:14:59] Tracking Nvidi

--- Profiling Block 39 / 39 ---


[codecarbon INFO @ 01:15:14] Energy consumed for RAM : 0.000042 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 01:15:14] Delta energy consumed for CPU with intel_rapl : 0.000498 kWh, power : 7.459956721518649 W
[codecarbon INFO @ 01:15:14] Energy consumed for All CPU : 0.000498 kWh
[codecarbon INFO @ 01:15:14] Energy consumed for all GPUs : 0.000172 kWh. Total GPU Power : 41.11380759079287 W
[codecarbon INFO @ 01:15:14] 0.000711 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 01:15:15] Energy consumed for RAM : 0.000044 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 01:15:15] Delta energy consumed for CPU with intel_rapl : 0.000024 kWh, power : 119.63641783982294 W
[codecarbon INFO @ 01:15:15] Energy consumed for All CPU : 0.000523 kWh
[codecarbon INFO @ 01:15:15] Energy consumed for all GPUs : 0.000180 kWh. Total GPU Power : 41.282069361836726 W
[codecarbon INFO @ 01:15:15] 0.000746 kWh of electricity and 0.000000 L of water were used since the beg

Error processing block 39: Job standard failed with exception: Expected the dataset block_39_k5_42_2a5d6d45-3c56-4d34-93f5-8cbfafe19f12 to have at most 500 dimensions, got 3947 instead. Consider using cat2vec to reduce the dimensionality of categorical variables.

=== Benchmark Complete. Summary saved to Benchmark/avatars/synthetic_blocks_k5_42/summary_stats_seed_42.csv ===
